# Post-correction for the transcribed lines of Cook journey using Mistral

In [1]:
# ── Cell 1: Imports ──────────────────────────────────────────────────
import json
import time
import requests
from pathlib import Path
from tqdm import tqdm

print("Imports done.")

Imports done.


In [2]:
# ── Cell 2: Paths ────────────────────────────────────────────────────
REPO_ROOT = Path.cwd()

# Input — folder of individual page transcription txt files
TRANSCRIPTIONS_DIR = REPO_ROOT / "transcriptions"

# Output — folder of corrected txt files (one per page, same naming)
CORRECTED_DIR = REPO_ROOT / "mistral_corrected_transcriptions"
CORRECTED_DIR.mkdir(parents=True, exist_ok=True)

# Ollama settings
OLLAMA_MODEL = "mistral-nemo"
OLLAMA_URL   = "http://localhost:11434/api/chat"

print(f"Input folder  : {TRANSCRIPTIONS_DIR}")
print(f"Output folder : {CORRECTED_DIR}")
print(f"Exists        : {TRANSCRIPTIONS_DIR.exists()}")

Input folder  : /Users/hedyeh/Ginger_Gradient/Capstone-Project/transcriptions
Output folder : /Users/hedyeh/Ginger_Gradient/Capstone-Project/mistral_corrected_transcriptions
Exists        : True


In [3]:
import re

def sort_key(p: Path):
    m = re.match(r'B(\d+)_P(\d+)', p.stem)
    return (int(m.group(1)), int(m.group(2))) if m else (99, 9999)

txt_files = sorted(TRANSCRIPTIONS_DIR.glob("*.txt"), key=sort_key)

# Build transcriptions dict: {page_id: [line1, line2, ...]}
transcriptions = {}
for txt_path in txt_files:
    page_id = txt_path.stem
    lines   = [
        line.rstrip()
        for line in txt_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    transcriptions[page_id] = lines

total_pages = len(transcriptions)
total_lines = sum(len(lines) for lines in transcriptions.values())

print(f"Pages loaded : {total_pages}")
print(f"Total lines  : {total_lines}")
print(f"\nBooks found  : {sorted(set(k[:2] for k in transcriptions))}")
print(f"\nSample page IDs:")
for pid in list(transcriptions.keys())[:8]:
    print(f"  {pid}  ({len(transcriptions[pid])} lines)")

# ── Limit to first 36 pages for testing ─────────────────────────────
transcriptions = dict(list(transcriptions.items())[:36])

print(f"\nLimited to {len(transcriptions)} pages for testing.")
print(f"Pages: {list(transcriptions.keys())}")

Pages loaded : 923
Total lines  : 31793

Books found  : ['B1', 'B2', 'B3', 'B4', 'B5', 'B6']

Sample page IDs:
  B1_P012  (14 lines)
  B1_P014  (31 lines)
  B1_P015  (27 lines)
  B1_P016  (27 lines)
  B1_P017  (25 lines)
  B1_P020  (25 lines)
  B1_P021  (23 lines)
  B1_P024  (24 lines)

Limited to 36 pages for testing.
Pages: ['B1_P012', 'B1_P014', 'B1_P015', 'B1_P016', 'B1_P017', 'B1_P020', 'B1_P021', 'B1_P024', 'B1_P025', 'B1_P028', 'B1_P029', 'B1_P030', 'B1_P031', 'B1_P034', 'B1_P035', 'B1_P038', 'B1_P039', 'B1_P042', 'B1_P043', 'B1_P046', 'B1_P047', 'B1_P050', 'B1_P051', 'B1_P052', 'B1_P053', 'B1_P056', 'B1_P057', 'B1_P060', 'B1_P061', 'B1_P064', 'B1_P065', 'B1_P068', 'B1_P069', 'B1_P072', 'B1_P073', 'B1_P074']


In [4]:
# ── Cell 4: Test Ollama connection ───────────────────────────────────
try:
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "stream": False,
            "messages": [
                {"role": "user", "content": "Reply with just: ready"}
            ]
        },
        timeout=30
    )
    print(f"Status : {response.status_code}")
    print(f"Reply  : {response.json()['message']['content'].strip()}")
    print("Ollama is running and model is loaded.")
except Exception as e:
    print(f"ERROR: {e}")
    print("Make sure 'ollama serve' is running in a terminal.")

Status : 200
Reply  : Ready
Ollama is running and model is loaded.


In [5]:
# ── Cell 5: System prompt and correction function ─────────────────────

def word_change_rate(original: str, corrected: str) -> float:
    """Fraction of words that changed between original and corrected."""
    orig_words = original.split()
    corr_words = corrected.split()
    if not orig_words:
        return 0.0
    changes = sum(
        1 for o, c in zip(orig_words, corr_words) if o != c
    )
    return changes / max(len(orig_words), len(corr_words))


SYSTEM_PROMPT = """You are a conservative OCR post-corrector for 18th century English manuscripts.

Your ONLY task is to fix clear character-level OCR errors caused by handwriting misreading.

CORRECT only these specific error types:
- Long-s confusion: 'f' used where 's' is clearly meant (e.g. 'fea' → 'sea', 'fhip' → 'ship', 'faw' → 'saw')
- Single character swaps: 'u'/'n', 'rn'/'m', 'cl'/'d', 'li'/'h', '1'/'l', '0'/'o'
- Clearly broken words with missing space (e.g. 'thewind' → 'the wind')

DO NOT change any of the following — return them exactly as given:
- Capitalisation — never change upper/lower case of any word
- Punctuation — do not add, remove, or change any punctuation
- Spelling — do not modernise or standardise any spelling
- Proper nouns — all place names, personal names, island names must be returned unchanged
- Archaic forms — 'shew', 'chuse', 'hath', 'whilst', 'ye', 'y' must stay as-is
- Abbreviations — 'Capt.', 'Sept.', 'Thermom.', 'Lat.' must stay as-is
- Latin words and phrases — return unchanged
- Any word you are uncertain about — return it unchanged

If you are not 100% certain a change fixes a clear OCR character error, return the word unchanged.
When in doubt, do nothing.

Return ONLY the corrected text with no explanation."""


def correct_line(text: str, retries: int = 3) -> str:
    """Send one line to Mistral-Nemo for post-correction."""
    # Skip very short lines — not enough context to correct reliably
    if len(text.strip()) < 5:
        return text

    for attempt in range(retries):
        try:
            response = requests.post(
                OLLAMA_URL,
                json={
                    "model": OLLAMA_MODEL,
                    "stream": False,
                    "options": {
                        "temperature": 0.0,   # fully deterministic
                        "num_predict": 200,
                    },
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {
                            "role": "user",
                            "content": (
                                f"OCR output: {text}\n\n"
                                f"Corrected text:"
                            )
                        }
                    ]
                },
                timeout=60
            )
            result = response.json()["message"]["content"].strip()

            # Safety 1: reject if result is much longer than input
            if len(result) > len(text) * 1.5:
                return text

            # Safety 2: reject if more than 30% of words changed
            # — sign of overcorrection or hallucination
            if word_change_rate(text, result) > 0.3:
                return text

            return result

        except Exception as e:
            if attempt < retries - 1:
                time.sleep(3)
            else:
                print(f"  [ERROR] {e} — keeping original")
                return text

    return text


# Quick test on a few sample lines
print("Testing correction function...\n")
samples = [
    "We faw feveral Albatrofes about the fhip",
    "The Thermometer at 76½ in my Cabin 44",
    "Sept. 9 18th In the Afternoon we faw Maurua",
    "from ulcatea",                          # should stay unchanged
    "Continuation of a journal",             # should stay unchanged
    "on board. his Magesties Ship",          # should fix
]
for s in samples:
    corrected = correct_line(s)
    changed   = "← changed" if corrected != s else "← unchanged"
    print(f"  IN : {s}")
    print(f"  OUT: {corrected}  {changed}")
    print()

Testing correction function...

  IN : We faw feveral Albatrofes about the fhip
  OUT: We faw feveral Albatrofes about the fhip  ← unchanged

  IN : The Thermometer at 76½ in my Cabin 44
  OUT: The thermometer at 76½ in my cabin 44  ← changed

  IN : Sept. 9 18th In the Afternoon we faw Maurua
  OUT: Sept. 9 18th In the Afternoon we faw Maurua  ← unchanged

  IN : from ulcatea
  OUT: from ulcatea  ← unchanged

  IN : Continuation of a journal
  OUT: Continuation of a journal  ← unchanged

  IN : on board. his Magesties Ship
  OUT: on board. his Magesties Ship  ← unchanged



In [6]:
# ── Cell 5b: Rule-based long-s fix ───────────────────────────────────
import re

LONG_S_FIXES = [
    (r'\bfaw\b',      'saw'),
    (r'\bfea\b',      'sea'),
    (r'\bfhip\b',     'ship'),
    (r'\bfhips\b',    'ships'),
    (r'\bfhore\b',    'shore'),
    (r'\bfome\b',     'some'),
    (r'\bfoon\b',     'soon'),
    (r'\bfaid\b',     'said'),
    (r'\bfail\b',     'sail'),
    (r'\bfailed\b',   'sailed'),
    (r'\bfouth\b',    'south'),
    (r'\bfeveral\b',  'several'),
    (r'\bfince\b',    'since'),
    (r'\bfide\b',     'side'),
    (r'\bfun\b',      'sun'),
    (r'\bfhew\b',     'shew'),
    (r'\bfhewed\b',   'shewed'),
    (r'\bftrong\b',   'strong'),
    (r'\bftill\b',    'still'),
    (r'\bftand\b',    'stand'),
]

def rule_based_fix(text: str) -> str:
    """Apply deterministic long-s corrections — no LLM needed."""
    for pattern, replacement in LONG_S_FIXES:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


# Quick test
print("Testing rule-based long-s fix:\n")
samples = [
    "We faw feveral Albatrofes about the fhip",
    "The Thermometer at 76½ in my Cabin 44",
    "we foon failed fouth toward the fhore",
    "from ulcatea",
]
for s in samples:
    fixed   = rule_based_fix(s)
    changed = "← changed" if fixed != s else "← unchanged"
    print(f"  IN : {s}")
    print(f"  OUT: {fixed}  {changed}")
    print()

Testing rule-based long-s fix:

  IN : We faw feveral Albatrofes about the fhip
  OUT: We saw several Albatrofes about the ship  ← changed

  IN : The Thermometer at 76½ in my Cabin 44
  OUT: The Thermometer at 76½ in my Cabin 44  ← unchanged

  IN : we foon failed fouth toward the fhore
  OUT: we soon sailed south toward the shore  ← changed

  IN : from ulcatea
  OUT: from ulcatea  ← unchanged



In [7]:
# ── Cell 6: Run post-correction on all lines ─────────────────────────
# Resume support: skip pages whose corrected .txt already exists

already_done = {p.stem for p in CORRECTED_DIR.glob("*.txt")}
pages_to_process = [
    pid for pid in transcriptions
    if pid not in already_done
]

print(f"Pages already corrected : {len(already_done)}")
print(f"Pages remaining         : {len(pages_to_process)}")
print(f"Total lines left        : {sum(len(transcriptions[pid]) for pid in pages_to_process)}")
print(f"\nStarting correction (temperature=0.0, model={OLLAMA_MODEL})...")
print("Tip: this runs locally — no API cost, no rate limits.\n")

for page_id in tqdm(pages_to_process, desc="Pages"):
    lines           = transcriptions[page_id]
    corrected_lines = []

    for line in lines:
        # Step 1 — LLM correction (conservative prompt)
        corrected = correct_line(line)

        # Step 2 — Rule-based long-s fix applied on top
        corrected = rule_based_fix(corrected)

        corrected_lines.append(corrected)

    # Save as individual .txt file — same name as input
    out_path = CORRECTED_DIR / f"{page_id}.txt"
    with open(out_path, "w", encoding="utf-8") as f:
        for line in corrected_lines:
            f.write(line + "\n")

    print(f"  Saved: {out_path.name}  ({len(corrected_lines)} lines)")

print(f"\nDone. {len(pages_to_process)} pages saved to: {CORRECTED_DIR}")

Pages already corrected : 0
Pages remaining         : 36
Total lines left        : 983

Starting correction (temperature=0.0, model=mistral-nemo)...
Tip: this runs locally — no API cost, no rate limits.



Pages:   3%|▎         | 1/36 [00:04<02:50,  4.87s/it]

  Saved: B1_P012.txt  (14 lines)


Pages:   6%|▌         | 2/36 [00:17<05:16,  9.30s/it]

  Saved: B1_P014.txt  (31 lines)


...

  Saved: B1_P072.txt  (33 lines)


Pages:  97%|█████████▋| 35/36 [07:02<00:13, 13.33s/it]

  Saved: B1_P073.txt  (27 lines)


Pages: 100%|██████████| 36/36 [07:16<00:00, 12.12s/it]

  Saved: B1_P074.txt  (33 lines)

Done. 36 pages saved to: /Users/hedyeh/Ginger_Gradient/Capstone-Project/mistral_corrected_transcriptions


In [8]:
# ── Quick summary of corrected output ────────────────────────────────
corrected_files = sorted(CORRECTED_DIR.glob("*.txt"))
print(f"Corrected pages saved : {len(corrected_files)}")
print(f"Output folder         : {CORRECTED_DIR}")
print(f"\nSample files:")
for f in corrected_files[:5]:
    print(f"  {f.name}")

# Preview first page — before and after
if corrected_files:
    first_page_id  = corrected_files[0].stem
    orig_path      = TRANSCRIPTIONS_DIR / f"{first_page_id}.txt"
    corr_path      = corrected_files[0]

    orig_lines = orig_path.read_text(encoding="utf-8").splitlines()
    corr_lines = corr_path.read_text(encoding="utf-8").splitlines()

    print(f"\n===== Sample comparison: {first_page_id} =====\n")
    changes_shown = 0
    for o, c in zip(orig_lines, corr_lines):
        if o != c:
            print(f"  BEFORE: {o}")
            print(f"  AFTER : {c}")
            print()
            changes_shown += 1
            if changes_shown >= 5:
                break
    if changes_shown == 0:
        print("  No changes on this page.")

Corrected pages saved : 36
Output folder         : /Users/hedyeh/Ginger_Gradient/Capstone-Project/mistral_corrected_transcriptions

Sample files:
  B1_P012.txt
  B1_P014.txt
  B1_P015.txt
  B1_P016.txt
  B1_P017.txt

===== Sample comparison: B1_P012 =====

  BEFORE: Journal of a journey
  AFTER : Journal of a Journey

  BEFORE: to the Cape of good Hope.
  AFTER : to the Cape of good Hope

  BEFORE: from July y.13½November
  AFTER : from July to November

